In [1]:
# 학습에 필요한 라이브러리를 불러옵니다.
# torchvision.models 에 EfficientNet-B4 가 포함되어 있습니다.
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import json
from datetime import datetime

# GPU 사용 가능 여부 확인
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.is_available()}")
print(f"GPU 이름: {torch.cuda.get_device_name(0)}")

PyTorch: 2.7.1+cu118
GPU: True
GPU 이름: NVIDIA GeForce RTX 2060 SUPER


In [3]:
# 학습에 필요한 모든 설정값을 한 곳에서 관리합니다.
# Experiment 1(ViT-Base)과 공정한 비교를 위해 학습 조건은 동일하게 유지합니다.
# 경로만 efficientnet 폴더로 분리해서 실험 결과가 섞이지 않도록 합니다.
CONFIG = {
    # 경로
    "data_dir": "../data/openfake",
    "save_dir": "../models/efficientnet",
    "checkpoint_path": "../models/efficientnet/checkpoint.pth",
    "best_model_path": "../models/efficientnet/best_model.pth",
    "history_path": "../models/efficientnet/history.json",

    # 모델
    "model_name": "efficientnet_b4",

    # 학습 하이퍼파라미터 (Experiment 1과 동일)
    "num_epochs": 10,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,

    # 기타
    "num_workers": 0,   # Windows 환경에서 DataLoader 행 방지
    "seed": 42,
}

os.makedirs(CONFIG["save_dir"], exist_ok=True)
torch.manual_seed(CONFIG["seed"])  # 재현성을 위한 시드 고정
print("설정 완료!")
print(f"저장 경로: {CONFIG['save_dir']}")

설정 완료!
저장 경로: ../models/efficientnet


In [4]:
# 이미지 파일 경로와 레이블(0=real, 1=fake)을 받아 PyTorch Dataset으로 만듭니다.
# 손상된 이미지가 있을 경우 검은 이미지로 대체해서 학습이 중단되지 않도록 합니다.
class OpenFakeDataset(Dataset):
    def __init__(self, file_list, labels, transform=None):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            # 손상된 이미지는 검은 이미지로 대체 (학습 중단 방지)
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label

print("Dataset 클래스 정의 완료!")

Dataset 클래스 정의 완료!


In [5]:
# real / fake 이미지 경로를 수집하고 train / val / test 로 분할합니다.
# seed를 고정해서 Experiment 1과 동일한 데이터 분할이 적용되도록 합니다.
data_dir = CONFIG["data_dir"]

# 파일 목록 수집 (real=0, fake=1)
real_files = [(f"{data_dir}/real/{f}", 0) for f in os.listdir(f"{data_dir}/real")]
fake_files = [(f"{data_dir}/fake/{f}", 1) for f in os.listdir(f"{data_dir}/fake")]

all_files = real_files + fake_files
np.random.seed(CONFIG["seed"])
np.random.shuffle(all_files)

files  = [f[0] for f in all_files]
labels = [f[1] for f in all_files]

# 8:1:1 비율로 분할
n = len(files)
n_train = int(n * CONFIG["train_ratio"])
n_val   = int(n * CONFIG["val_ratio"])

train_files  = files[:n_train]
train_labels = labels[:n_train]
val_files    = files[n_train:n_train+n_val]
val_labels   = labels[n_train:n_train+n_val]
test_files   = files[n_train+n_val:]
test_labels  = labels[n_train+n_val:]

print(f"전체: {n}장")
print(f"train: {len(train_files)}장")
print(f"val:   {len(val_files)}장")
print(f"test:  {len(test_files)}장")
print(f"real 비율 (전체): {labels.count(0)/len(labels)*100:.1f}%")
print(f"fake 비율 (전체): {labels.count(1)/len(labels)*100:.1f}%")

전체: 128444장
train: 102755장
val:   12844장
test:  12845장
real 비율 (전체): 38.9%
fake 비율 (전체): 61.1%


In [6]:
# EfficientNet-B0 권장 입력 크기: 224×224
# ImageNet 정규화 값 사용 (torchvision 표준)
# Resize(256) → CenterCrop(200) → Resize(224) 순서로 테두리를 잘라냅니다.
# 워터마크가 이미지 우측 하단에 존재할 수 있어 shortcut learning 방지를 위해 적용합니다.
# train에는 증강을 적용하고, val/test에는 적용하지 않습니다.
IMG_SIZE      = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),                                             # 약간 크게 리사이즈
    transforms.CenterCrop((200, 200)),                                         # 테두리 잘라내기 (워터마크 제거)
    transforms.Resize((IMG_SIZE, IMG_SIZE)),                                   # 모델 입력 크기로 다시 맞추기
    transforms.RandomHorizontalFlip(),                                         # 좌우 반전
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),     # 색상 변화
    transforms.RandomRotation(10),                                             # 최대 10도 회전
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((200, 200)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = OpenFakeDataset(train_files, train_labels, transform=train_transform)
val_dataset   = OpenFakeDataset(val_files,   val_labels,   transform=val_transform)
test_dataset  = OpenFakeDataset(test_files,  test_labels,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"])
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print(f"train batches: {len(train_loader)}")
print(f"val batches:   {len(val_loader)}")
print(f"test batches:  {len(test_loader)}")

train batches: 6423
val batches:   803
test batches:  803


In [7]:
# EfficientNet-B4를 ImageNet 사전학습 가중치로 불러옵니다.
# 마지막 분류 레이어만 1000 클래스 → 2 클래스(real/fake)로 교체합니다.
# real이 더 적으므로(50K vs 80K) 클래스 가중치로 불균형을 보정합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# 사전학습 가중치 로드
# model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)

# 분류 레이어 교체: in_features(1792) → 2
# num_features = model.classifier[1].in_features
# model.classifier[1] = nn.Linear(num_features, 2)
# model = model.to(device)

# EfficientNet-B4 → B0으로 교체 (VRAM 부족으로 인한 변경)
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# 분류 레이어 교체
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model = model.to(device)



optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

# 클래스 불균형 보정: real이 적을수록 real에 더 높은 가중치 부여
n_real = labels.count(0)
n_fake = labels.count(1)
weight = torch.tensor([n_fake / n_real, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=weight)

print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")
print(f"클래스 가중치 - real: {weight[0]:.3f}, fake: {weight[1]:.3f}")

device: cuda
모델 파라미터: 4,010,110
클래스 가중치 - real: 1.569, fake: 1.000


In [8]:
# 체크포인트 저장/불러오기, 에폭 학습, 검증 함수를 정의합니다.
# 중간에 학습이 끊겨도 이어받을 수 있도록 매 에폭 후 체크포인트를 저장합니다.
# torchvision 모델은 model(images) 로 바로 logits을 반환합니다. (ViT와 다른 부분)

def save_checkpoint(epoch, model, optimizer, scheduler, history, best_val_acc):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
    }, CONFIG["checkpoint_path"])
    with open(CONFIG["history_path"], "w") as f:
        json.dump(history, f)
    print(f"체크포인트 저장: epoch {epoch}")

def load_checkpoint(model, optimizer, scheduler):
    if not os.path.exists(CONFIG["checkpoint_path"]):
        print("체크포인트 없음 - 처음부터 학습 시작")
        return 0, {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}, 0.0

    checkpoint = torch.load(CONFIG["checkpoint_path"], map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    start_epoch  = checkpoint["epoch"] + 1
    best_val_acc = checkpoint["best_val_acc"]

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    if os.path.exists(CONFIG["history_path"]):
        with open(CONFIG["history_path"]) as f:
            history = json.load(f)

    print(f"체크포인트 로드 완료 - epoch {checkpoint['epoch']}부터 이어서 학습")
    return start_epoch, history, best_val_acc

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    log_interval = max(1, len(loader) // 10)  # 10% 간격으로 배치 로그 출력

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        logits = model(images)          # torchvision: 텐서 직접 반환 (ViT와 다른 부분)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        if batch_idx % log_interval == 0:
            print(f"  batch {batch_idx}/{len(loader)} | loss: {loss.item():.4f}")

    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / len(loader), correct / total, f1

print("학습 함수 정의 완료!")

학습 함수 정의 완료!


In [ ]:
# 실제 학습을 실행합니다.
# CosineAnnealingLR로 학습률을 점진적으로 감소시켜 수렴을 돕습니다.
# val acc가 개선될 때만 best_model.pth를 갱신하고, 매 에폭 체크포인트를 저장합니다.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["num_epochs"]
)

# 체크포인트가 있으면 이어서, 없으면 처음부터 시작
start_epoch, history, best_val_acc = load_checkpoint(model, optimizer, scheduler)

training_start_time = datetime.now()
print(f"\n학습 시작: epoch {start_epoch} ~ {CONFIG['num_epochs']-1}")
print(f"학습 시작 시각: {training_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*50)

for epoch in range(start_epoch, CONFIG["num_epochs"]):
    print(f"\n[Epoch {epoch+1}/{CONFIG['num_epochs']}]")
    start_time = datetime.now()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_f1 = validate(model, val_loader, criterion, device)
    scheduler.step()

    elapsed       = (datetime.now() - start_time).seconds // 60
    total_elapsed = (datetime.now() - training_start_time).seconds // 60
    print(f"  train loss: {train_loss:.4f} | train acc: {train_acc*100:.2f}%")
    print(f"  val loss:   {val_loss:.4f} | val acc:   {val_acc*100:.2f}% | val F1: {val_f1:.4f}")
    print(f"  에폭 소요시간: {elapsed}분 | 전체 경과: {total_elapsed}분")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CONFIG["best_model_path"])
        print(f"  ✅ 최고 모델 저장! val acc: {val_acc*100:.2f}%")

    save_checkpoint(epoch, model, optimizer, scheduler, history, best_val_acc)

print("\n학습 완료!")
print(f"최고 val acc: {best_val_acc*100:.2f}%")
print(f"총 학습 시간: {(datetime.now() - training_start_time).seconds // 60}분")

체크포인트 없음 - 처음부터 학습 시작

학습 시작: epoch 0 ~ 9
학습 시작 시각: 2026-04-13 20:10:49

[Epoch 1/10]
  batch 0/6423 | loss: 0.6652


In [ ]:
# 학습 중 가장 높은 val acc를 기록한 모델(best_model.pth)로 테스트셋을 평가합니다.
# Accuracy, F1, AUC-ROC 세 가지 지표로 Experiment 1과 비교합니다.
model.load_state_dict(torch.load(CONFIG["best_model_path"], map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)[:, 1]  # fake 클래스 확률 (AUC 계산용)
        preds  = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
test_f1  = f1_score(all_labels, all_preds, average="macro")
test_auc = roc_auc_score(all_labels, all_probs)

print("=" * 40)
print("[Experiment 2 - EfficientNet-B4 Test 결과]")
print(f"  Accuracy : {test_acc*100:.2f}%")
print(f"  F1 Score : {test_f1:.4f}")
print(f"  AUC-ROC  : {test_auc:.4f}")
print("=" * 40)
print()
print(classification_report(all_labels, all_preds, target_names=["real", "fake"]))

In [ ]:
# 학습 과정에서 저장한 history.json을 불러와 loss와 accuracy 곡선을 그립니다.
# 그래프를 이미지로 저장해두면 포트폴리오에 바로 사용할 수 있습니다.
with open(CONFIG["history_path"], "r") as f:
    history = json.load(f)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"],   label="val")
plt.title("Loss — EfficientNet-B4")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot([a*100 for a in history["train_acc"]], label="train")
plt.plot([a*100 for a in history["val_acc"]],   label="val")
plt.title("Accuracy (%) — EfficientNet-B4")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/training_curve.png", dpi=150)
plt.show()
print("그래프 저장 완료!")